In [1]:
pip install pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [10]:
import json
from pathlib import Path

import pandas as pd
from sklearn.metrics import cohen_kappa_score


# ============================================================
# 1. SETTINGS
# ============================================================

from pathlib import Path

# HUMAN1_FOLDER = Path("annotator1_folder")
# HUMAN2_FOLDER = Path("annotator2_folder")
# HUMAN1_FOLDER = Path.cwd().parent /"Case_Study_JSONs"
# HUMAN2_FOLDER = Path.cwd().parent /"Case_Study_JSONs_Ollama"

# HUMAN1_FOLDER = Path.cwd().parent /"coded_json" #gpt coded data is there
# HUMAN2_FOLDER = Path.cwd().parent /"agreed_json"


HUMAN1_FOLDER = Path.cwd().parent /"Llamacoded_json" #Llama coded data is there
HUMAN2_FOLDER = Path.cwd().parent /"agreed_json"

# HUMAN1_FOLDER = Path("secondround/annotator1CM_folder")
# HUMAN2_FOLDER = Path("secondround/annotator2RR_folder")

# Output files
# MATRIX_OUTPUT = "human1_human2_matrix.csv"
# KAPPA_OUTPUT = "human1_human2_kappa.csv"
# MATRIX_OUTPUT = "gpt_ollama_matrix.csv"
# KAPPA_OUTPUT = "gpt_ollama_kappa.csv"


# MATRIX_OUTPUT = "GPT_Agreed.csv"
# KAPPA_OUTPUT = "GPT_Agreed_kappa.csv"

MATRIX_OUTPUT = "Llama_Agreed.csv"
KAPPA_OUTPUT = "Llama_Agreed_kappa.csv"

# ============================================================
# 2. DEFINE THE VARIABLES IN YOUR SCHEMA
# ============================================================

# Single-value categorical variables
# SINGLE_VALUE_FIELDS = {
#     "catalyst.precipitating_event",

#     "attack_characteristics.attack",
#     "attack_characteristics.attack_objective",
# }


# Multi-value categorical variables
# These are arrays in your JSON schema.
MULTI_VALUE_FIELDS = {
    "catalyst.precipitating_event",
    "actor_characteristics.psychological_state",
    "actor_characteristics.personality_characteristics",
    "actor_characteristics.attitude_towards_work",
    "actor_characteristics.motivation_to_attack",
    "actor_characteristics.skill_set",
    "actor_characteristics.opportunity",
    "actor_characteristics.historical_behaviour",
    "actor_characteristics.observed_physical_behaviour",
    "actor_characteristics.observed_cyber_behaviour",
    
    "attack_characteristics.attack",
    "attack_characteristics.attack_objective",
    "attack_characteristics.attack_step",
    "attack_characteristics.attack_step_goal",

    "organisation_characteristics.asset",
    "organisation_characteristics.vulnerability",
}


# Actor-level fields
# These require special handling because "actors" is itself an array
# containing objects.
ACTOR_FIELDS = {
    "type_of_actor",
    "enterprise_role",
    "state_of_relationship",
}


# ============================================================
# 3. LOAD JSON
# ============================================================

def load_json(path):
    """Load a JSON file."""
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def get_nested_value(data, field_path):
    """
    Retrieve a value from nested JSON using dot notation.

    Example:
        get_nested_value(data, "attack_characteristics.attack")

    returns:
        data["attack_characteristics"]["attack"]
    """

    value = data

    for key in field_path.split("."):
        value = value[key]

    return value



CATEGORY_MAPPING = {
    "work_place_rule_violation": "workplace_rule_violation",
    "privilege/permission_abus": "privilege/permission_abuse",
    "disputes_with_employers;": "disputes_with_employers",
    "organisational_assest":"organisational_asset",
    "access/_access_acquisition":"access/access_acquisition",
    "collection/_acquisition":"collection/acquisition",
    "concealment/_evasion":"concealment/evasion",
    "data_transfer/_exfiltration":"data_transfer/exfiltration",
    "execution/_action":"execution/action",
    "information_gathering_/_discovery":"information_gathering/discovery",
    "persistence/_continued_access":"persistence/continued_access",
    "privilege/_permission_abuse":"privilege/permission_abuse"

}

# ============================================================
# NORMALISE CATEGORY
# ============================================================
def normalize_category(value):
    """
    Normalize category values and map known variants
    to a single canonical category.
    """

    if value is None:
        return None

    value = str(value).strip().lower()

    # Basic formatting normalization
    value = value.replace(" ", "_")
    value = value.replace("-", "_")

    while "__" in value:
        value = value.replace("__", "_")

    value = value.strip("_")

    # Map known variants to canonical values
    value = CATEGORY_MAPPING.get(value, value)

    return value


# ============================================================
# NORMALISE
# ============================================================


def ensure_list(value):
    """
    Make sure a multi-value field is always represented as a list.

    - None       -> []
    - list       -> unchanged
    - string     -> [string]
    - other      -> [value]
    """

    if value is None:
        return []

    if isinstance(value, list):
        return value

    if isinstance(value, str):
        return [value]

    return [value]

# ============================================================
# 4. GET ALL CASE IDs
# ============================================================

human1_files = {
    path.stem: path
    for path in HUMAN1_FOLDER.glob("*.json")
}

print(human1_files)

human2_files = {
    path.stem: path
    for path in HUMAN2_FOLDER.glob("*.json")
}


case_ids = sorted(
    set(human1_files.keys()) &
    set(human2_files.keys())
)

if not case_ids:
    raise ValueError(
        "No matching JSON case files were found in human1 and human2 folders."
    )

print(f"Found {len(case_ids)} cases coded by both humans.")


# ============================================================
# 5. LOAD ALL CASES
# ============================================================

human1_data = {}
human2_data = {}

for case_id in case_ids:
    human1_data[case_id] = load_json(human1_files[case_id])
    human2_data[case_id] = load_json(human2_files[case_id])


# ============================================================
# 6. CREATE THE AGREEMENT MATRIX
# ============================================================

matrix_rows = []


# ------------------------------------------------------------
# 6B. Multi-value fields
# ------------------------------------------------------------

for field in MULTI_VALUE_FIELDS:

    # Get every normalized category appearing
    # in either coder's data
    all_categories = set()

    for case_id in case_ids:

        h1_values = [
            normalize_category(value)
            for value in ensure_list(
                get_nested_value(
                    human1_data[case_id],
                    field
                )
            )
        ]

        h2_values = [
            normalize_category(value)
            for value in ensure_list(
                get_nested_value(
                    human2_data[case_id],
                    field
                )
            )
        ]

        # Remove empty / invalid values
        h1_values = {
            value for value in h1_values
            if value
        }

        h2_values = {
            value for value in h2_values
            if value
        }

        all_categories.update(h1_values)
        all_categories.update(h2_values)

    # Create one binary variable for every category
    for category in sorted(all_categories):

        for case_id in case_ids:

            h1_values = {
                normalize_category(value)
                for value in ensure_list(
                    get_nested_value(
                        human1_data[case_id],
                        field
                    )
                )
                if value
            }

            h2_values = {
                normalize_category(value)
                for value in ensure_list(
                    get_nested_value(
                        human2_data[case_id],
                        field
                    )
                )
                if value
            }

            h1_binary = int(category in h1_values)
            h2_binary = int(category in h2_values)

            matrix_rows.append({
                "case_id": case_id,
                "field": field,
                "category": category,
                "human1": h1_binary,
                "human2": h2_binary,
                "agreement": int(h1_binary == h2_binary)
            })


# ============================================================
# 7. CONVERT TO DATAFRAME
# ============================================================

agreement_matrix = pd.DataFrame(matrix_rows)


# Save the complete case-by-case matrix
agreement_matrix.to_csv(
    MATRIX_OUTPUT,
    index=False,
    encoding="utf-8-sig"
)

print(f"\nAgreement matrix saved to: {MATRIX_OUTPUT}")


# ============================================================
# 8. CALCULATE COHEN'S KAPPA
# ============================================================

kappa_results = []


for (field, category), group in agreement_matrix.groupby(
    ["field", "category"]
):

    h1 = group["human1"].tolist()
    h2 = group["human2"].tolist()

    # Cohen's kappa
    kappa = cohen_kappa_score(h1, h2)

    # Raw percentage agreement
    percent_agreement = (
        sum(a == b for a, b in zip(h1, h2))
        / len(h1)
        * 100
    )

    kappa_results.append({
        "field": field,
        "category": category,
        "n_cases": len(group),
        "human1_positive": sum(h1),
        "human2_positive": sum(h2),
        "percent_agreement": round(percent_agreement, 2),
        "cohens_kappa": round(kappa, 4)
    })


# ============================================================
# 9. RESULTS TABLE
# ============================================================

kappa_results_df = pd.DataFrame(kappa_results)


# Sort results
kappa_results_df = kappa_results_df.sort_values(
    ["field", "category"]
)


# Save
kappa_results_df.to_csv(
    KAPPA_OUTPUT,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 10. PRINT RESULTS
# ============================================================

print("\n" + "=" * 80)
print("HUMAN 1 vs HUMAN 2 — COHEN'S KAPPA")
print("=" * 80)

print(
    kappa_results_df.to_string(index=False)
)

print("\nResults saved to:")
print(KAPPA_OUTPUT)

{'CS_001': WindowsPath('C:/Users/Nadeeka_92/Experiments/Llamacoded_json/CS_001.json'), 'CS_002': WindowsPath('C:/Users/Nadeeka_92/Experiments/Llamacoded_json/CS_002.json'), 'CS_003': WindowsPath('C:/Users/Nadeeka_92/Experiments/Llamacoded_json/CS_003.json'), 'CS_004': WindowsPath('C:/Users/Nadeeka_92/Experiments/Llamacoded_json/CS_004.json'), 'CS_005': WindowsPath('C:/Users/Nadeeka_92/Experiments/Llamacoded_json/CS_005.json'), 'CS_006': WindowsPath('C:/Users/Nadeeka_92/Experiments/Llamacoded_json/CS_006.json'), 'CS_008': WindowsPath('C:/Users/Nadeeka_92/Experiments/Llamacoded_json/CS_008.json'), 'CS_010': WindowsPath('C:/Users/Nadeeka_92/Experiments/Llamacoded_json/CS_010.json'), 'CS_012': WindowsPath('C:/Users/Nadeeka_92/Experiments/Llamacoded_json/CS_012.json'), 'CS_018': WindowsPath('C:/Users/Nadeeka_92/Experiments/Llamacoded_json/CS_018.json')}
Found 10 cases coded by both humans.

Agreement matrix saved to: Llama_Agreed.csv

HUMAN 1 vs HUMAN 2 — COHEN'S KAPPA
                     

C:\Users\Nadeeka_92\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
C:\Users\Nadeeka_92\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
C:\Users\Nadeeka_92\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
C:\Users\Nadeeka_92\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
